In [ ]:
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(ggdist)
library(ggforce)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(purrr)
library(broom)
library(Polychrome)
library(cowplot)
library(reshape2)
library(doParallel)

In [ ]:
metadata <- read.csv("DOME-Final-16S-Nasal-Metadata.csv")
metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

GENUS <- read.csv("DOME-16S-Taxa-Counts-Genus.csv")

asv_counts <- GENUS %>% dplyr::rename(Sample.ID = X)

relative_abundance_genus <- asv_counts %>%
  tibble::column_to_rownames("Sample.ID") %>%
  { 
    rs <- rowSums(.)
    rs[rs == 0] <- 1  # prevent divide by zero
    sweep(., 1, rs, "/")
  } %>%
  as.data.frame() %>%
  tibble::rownames_to_column("Sample.ID")

# Step 3: Apply 0.05% cutoff
relative_abundance_genus <- relative_abundance_genus %>%
  dplyr::mutate(across(-Sample.ID, ~ ifelse(.x <= 0.0005, 0, .x)))

# Step 4: Merge with metadata
merged_GENUS <- right_join(metadata, relative_abundance_genus, by = "Sample.ID")

merged_GENUS_long <- merged_GENUS %>%
  pivot_longer(
    cols = where(is.numeric),
    names_to = "Genus",
    values_to = "Abundance"
  ) %>%
  mutate(Genus = as.character(Genus))   # <- prevent numeric factors

# 1. Identify top 20 genera by abundance
genus_order <- merged_GENUS_long %>%
  group_by(Genus) %>%
  summarise(total_abundance = sum(Abundance, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(total_abundance)) %>%
  slice_head(n = 20) %>%
  pull(Genus)

# 2. Define "Other" for everything not in top genera
merged_GENUS_long <- merged_GENUS_long %>%
  mutate(
    Genus = as.character(Genus),   # <-- force character names
    Genus = ifelse(Genus %in% genus_order, Genus, "Other"),
    Genus = factor(Genus, levels = c(genus_order, "Other"))
  )

top_20_long <- relative_abundance_genus %>%
  pivot_longer(-Sample.ID, names_to = "Genus", values_to = "Abundance") %>%
  mutate(Genus = as.character(Genus)) %>%
  # Recode Other
  mutate(Genus = ifelse(Genus %in% genus_order, Genus, "Other")) %>%
  group_by(Sample.ID, Genus) %>%
  summarise(Abundance = sum(Abundance, na.rm = TRUE), .groups = "drop") %>%
  ungroup() %>%
  # Only after summing, convert to percent
  mutate(
    Abundance = Abundance * 100,
    Genus = factor(Genus, levels = c(genus_order, "Other"))
  ) %>%
  left_join(metadata, by = "Sample.ID")

genus_colors <- c("Acinetobacter" = "#C6946C",
                  "Aerococcus" = "#C09B00",
                  "Arthrobacter_B" = "#BA9F00",
                  "Bacillus" = "#32D4C7",
                  "Bacillus_A" = "#B1A300",
                  "Brachybacterium" = "#C9CB81",
                  "Brevibacillus" = "#BDCD84",
                  "Brevibacterium" = "#C3CC7E",
                  "Cellulosimicrobium" = "#9F00BB",
                  "Citricoccus" = "#7EAD00",
                  "Corynebacterium" = "#93D19C",
                  "Cytobacillus" = "#00B27D",
                  "Dolosigranulum" = "#00B393",
                  "Kocuria" = "#DCB9E6",
                  "Lysinibacillus" = "#B4006C",
                  "Mammaliicoccus" = "#D33639",
                  "Microbacterium" = "#00B2BA",
                  "Micrococcus" = "#89D2A1",
                  # "Psychrobacter" = "#FDE4CF",
                  "Nocardiopsis" = "#E8B2D5",
                  "Paenibacillus" = "#00AFC2",
                  "Priestia" = "#00ACC9",
                  "Shouchella" = "#B3C6E9",
                  "Staphylococcus" = "#E8B8B5",
                  "Streptomyces" = "#00A7CD",
                  "Virgibacillus" = "#91A0D0",
                  "Other" = "#B5B5B2",
                    "Cow" = "#990006",
                    "Non-Farmer" = "#B5B5B2", 
                    "Farmer" = "#386EC2")

# Find all genera that need colors
all_genera <- unique(top_20_long$Genus)

missing_genera <- setdiff(all_genera, names(genus_colors))

extra_colors <- createPalette(length(missing_genera), c("#00ACC9"), M = 50000)
names(extra_colors) <- missing_genera
genus_colors <- c(genus_colors, extra_colors)

genus_colors["Other"] <- "#B5B5B2"

In [ ]:
top_20_long <- top_20_long %>%
mutate(
    Genus = factor(Genus),
    Month = factor(Month),
    Season = factor(Season, levels = c("Spring", "Summer", "Autumn", "Winter")),
    Year = factor(Year),
    Cohort = factor(Cohort)
)

In [ ]:
top_20_agg <- top_20_long %>%
  group_by(Year, Season, Cohort, Genus) %>%
  summarise(Abundance = sum(Abundance), .groups = "drop") %>%
  group_by(Year, Season, Cohort) %>%
  mutate(Abundance = Abundance / sum(Abundance) * 100) %>%  # normalize to 100%
  ungroup()

In [ ]:
relative_abundance_plot <- ggplot(top_20_agg, aes(x = Cohort, y = Abundance, fill = Genus)) +
  geom_bar(stat = "identity", position = "stack") +
  facet_grid2(
    Season ~ Year,
    scales = "free_x",
    strip = ggh4x::strip_vanilla()
  ) +
  scale_y_continuous(name = "Relative Abundance (%)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    strip.background = element_rect(fill = "black"),
    strip.text = element_text(color = "white", face = "bold", size = 8),
    axis.text.x = element_text(size = 8),
    axis.title.x = element_blank(),
    axis.text.y.left = element_text(size = 8),
    axis.ticks.y.left = element_line(),
    axis.text.y.right = element_blank(),
    axis.ticks.y.right = element_blank(),
    legend.position = "right",
    legend.key.size = unit(0.8, "lines"),
    legend.text = element_text(size = 8),
    panel.spacing = unit(1.2, "lines"),
    text = element_text(family = "Helvetica", size = 12)
  ) +
  xlab("") +
  scale_fill_manual(
    values = genus_colors,
    drop = FALSE,
    guide = guide_legend(
      ncol = 1,
      byrow = TRUE,
      direction = "vertical"
    )
  )


In [ ]:
ggplot2::ggsave("16S-Season-Year-Relative-Abundance.pdf", relative_abundance_plot, dpi = 360, width = 12, height = 8, units = "in")